In [ ]:
from trainer import BaseTrainModule, Trainer
from model import JSWRegression
from dataset_pre import Data_Loader

import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
import matplotlib.pyplot as plt
from torch import optim
import torch.nn as nn
import os

ROOT_PATH = os.path.abspath(os.path.join(os.path.join(os.path.join(os.getcwd(), os.pardir), os.pardir), os.pardir))

class MyTrainModule(BaseTrainModule):
    def __init__(self, reduce=100):
        super().__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.reduce = reduce
        print(self.reduce)
        self.model = JSWRegression(in_channels=1, out_channels=1)
        self.model = nn.DataParallel(self.model)
        self.model.to(device=self.device)

    def configure_lossfunctions(self):
        self.criterion = nn.MSELoss()

    def configure_optimizers(self, lr):
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr, betas=(0.5, 0.999))

        return self.optimizer

    def configure_scheduler(self, optimizers):
        optimizer = optimizers
        self.scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=100, gamma=0.5)

        return (self.scheduler)

    def configure_logs(self):
        # learning rate
        self.set_log(name='Lr', obj=self.optimizer, category='lr')
        # loss function
        self.set_log(name='Loss_jsw', obj=self.criterion, category='loss', mode='train')
        self.set_log(name='Loss_jsw_val', obj=self.criterion, category='loss', mode='valid')

    def training_step(self, batch_idx, batch):
        self.model.train()
        image, jsw = batch

        self.optimizer.zero_grad()
        pre_jsw = self.model(image)
        print('{}, Pre: {}, GT: {}'.format(self.reduce, pre_jsw.cpu().detach().numpy()[0][0], jsw.cpu().detach().numpy()[0][0]))
        
        loss = torch.sqrt(self.criterion(pre_jsw, jsw))

        loss.backward()
        self.optimizer.step()

        return loss

    def validation_step(self, batch_idx, batch):
        self.model.train()
        image, jsw = batch

        self.optimizer.zero_grad()
        pre_jsw = self.model(image)
        
        loss_val = torch.sqrt(self.criterion(pre_jsw, jsw))
        print('Pre: {}, GT: {}'.format(pre_jsw.cpu().detach().numpy()[0][0], jsw.cpu().detach().numpy()[0][0]))

        image_list = [image]

        return loss_val, image_list

    def configure_saveprocess(self):
        self.set_save_parameter(model=self.model, loss_name='Loss_jsw',
                                save_path=ROOT_PATH + '/experiments/Exp_downstream/JSW_evaluation/parameter/best_jsw_model_pre_{}.pth'.format(self.reduce))

    def show_single_log_image(self, image_box):
        image = image_box
        plt.imshow(image[0][0])
        plt.show()


if __name__ == "__main__":
    # Data Loading
    image_size = 256
    transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(0, 1)
                                    ])

    reduce_list = [10]

    for REDUCE in reduce_list:
        print(REDUCE)
        dataset = Data_Loader(ROOT_PATH + '/experiments/Exp_downstream/JSW_evaluation/pre_JSW_data_{}.json'.format(REDUCE), transform)
        train_size = int(0.8 * len(dataset)) 
        test_size = len(dataset) - train_size  
        train_dataset, valid_dataset = random_split(dataset, [train_size, test_size])

        mytrainmodule = MyTrainModule(reduce=REDUCE)
        trainer = Trainer(train_module=mytrainmodule, train_dataset=train_dataset, valid_dataset=valid_dataset)
        trainer.configure(batch_size=48, epochs=150, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                        result_number=1, lr=1e-4)
        trainer.fit()
